In [ ]:
session.sql("DROP TABLE IF EXISTS tmp_allowable_values_stream_snapshot").collect()
print("Temp table dropped")

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_allowable_values_stream_snapshot AS
SELECT *
FROM STREAM_T_ALLOWABLE_VALUES
WHERE METADATA$ACTION = 'INSERT'
""").collect()

av_raw = session.table("tmp_allowable_values_stream_snapshot")
av_raw = av_raw.drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID")
print(f"snapshot allowable_values records found")

In [ ]:
ind_columns = ["VALUE_DEFAULT_IND"]

av_clean = av_raw
for c in ind_columns:
    av_clean = av_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("N")
        ).otherwise(trim(col(c)))
    )

code_columns = ["VALUE_CODE"]

for c in code_columns:
    av_clean = av_clean.with_column(
        c,
        upper(trim(col(c)))
    )

desc_columns = ["VALUE_SHORT_DESC", "VALUE_LONG_DESC"]

for c in desc_columns:
    av_clean = av_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("NA")
        ).otherwise(trim(col(c)))
    )

user_columns = ["ADD_USER", "MOD_USER"]

for c in user_columns:
    av_clean = av_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("SYSTEM")
        ).otherwise(trim(col(c)))
    )

trim_only_columns = ["MINIMUM_NUM", "MAXIMUM_NUM", "SOURCE_FILE_NAME"]

for c in trim_only_columns:
    av_clean = av_clean.with_column(
        c,
        trim(col(c))
    )

av_clean = av_clean.with_column_renamed("LOAD_TIMESTAMP", "RAW_LOAD_TIMESTAMP")
print("av clean")

In [ ]:
valid_av = av_clean.filter(
    col("AV_ID").is_not_null()
)

invalid_av = av_clean.filter(
    col("AV_ID").is_null()
).with_column(
    "ERROR_REASON",
    lit("AV_ID_NULL")
)
print("seperated valid and invalid records")

In [ ]:
av_final = valid_av.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

av_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_ALLOWABLE_VALUES}",
    mode="truncate"
)

print(f"AV loaded into {STG}.{STG_ALLOWABLE_VALUES}")

In [ ]:
invalid_av.create_or_replace_temp_view("tmp_invalid_av")

invalid_count = invalid_av.count()

if invalid_count > 0:
    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
        'T_ALLOWABLE_VALUES',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_av
    """).collect()
    print(f"Invalid records saved into {STG}.{INVALID_RECORDS}")
else:
    print("No invalid records")

In [ ]:
rows_processed = av_final.count()
rows_failed = invalid_count

status = 'SUCCESS' if rows_failed == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    'STG_ALLOWABLE_VALUES_LOAD',
    'STAGING',
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    status,
    None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status,
    'STG_ALLOWABLE_VALUES_LOAD',
    'STAGING',
    f'ALLOWABLE_VALUES staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}'
)
print("Audit log inserted and email sent")